In [ ]:
from pathlib import Path
import os
from neuralhydrology.evaluation import metrics, get_tester
from neuralhydrology.utils.config import Config
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pandas as pd
import yaml
import warnings
warnings.simplefilter("ignore", FutureWarning)
import plotly.graph_objects as go


In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go

Allereerst bepalen wij bij welke epoch de modellen de hoogste NSE hebben, daarna plotten we de scores

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import os 
def extract_tensorboard_scalars(logdir):
    event_acc = EventAccumulator(logdir)
    event_acc.Reload()  # Load the TensorBoard logs

    scalars = {}
    for tag in event_acc.Tags()['scalars']:  # Get all scalar tags
        scalars[tag] = [(e.step, e.value) for e in event_acc.Scalars(tag)]
    
    return scalars

Hieronder base_folder aanpassen naar je eigen mapje.

In [ ]:
base_folder = r'C:\Users\christiaan.wewer\Documents\projecten\Z0264 - Neural Hydrology - Onderzoek Afvoerverwachtingen HDSR o.b.v. AI\Gegevens\Bewerking\Scripts\visualisatie_runs'

In [ ]:
r'C:\Users\christiaan.wewer\Documents\projecten\Z0264 - Neural Hydrology - Onderzoek Afvoerverwachtingen HDSR o.b.v. AI\Gegevens\Bewerking\Scripts\visualisatie_runs\runs\development_run_1303_145926'

In [ ]:
runs_folder = r'runs'
run_names = ['development_run_1303_145926', 'development_run_1303_180110'] # later in de grafieken ook wel model 1 en 2 genoemd, respectivelijk

In [ ]:
# get NSE curves of validation data

for r in run_names:
    folder = f'{base_folder}\{runs_folder}\{r}'
    data = extract_tensorboard_scalars(folder)
    max_nse_epoch = (np.argmax([score for epoch, score in data['valid/mean_nse_1d']])+1)*5 # +1 because of 0 indexing, * 5 because we save per 5 epochs
    print(f'De maximale NSE van model {r} is bij epoch {max_nse_epoch}')


Neuralhydrology pakt van zichzelf altijd het model van de laatste epoch beschikbaar, maar dat is wellicht niet het model met de hoogste NSE score op de validatie set.
Daarom checken wij eerst welke epoch de hoogste NSE heeft en kopieren wij handmatig alle model_epochXXX.pt en optimizer_state_epochXXX.pt naar het backup mapje. Hierbij is XXX de modellen met epochs die hoger zijn dan aangegeven in de print hierboven.

In [ ]:

run_type = ['validation', 'test']
results_list = []
for run in run_names:
    results_val_test = []
    for rt in run_type:
        run_folder_path = Path(runs_folder + '/' + run)
        with open(run_folder_path / 'config.yml') as file:
            config = yaml.load(file, Loader=yaml.FullLoader)
        config['device'] = 'cpu'
        tester_config = Config(config)
        tester = get_tester(tester_config, run_folder_path, init_model=True, period=rt)
        results = tester.evaluate(save_results=True, metrics=tester_config.metrics)
        results_val_test.append(results)
    results_list.append(results_val_test)


In [ ]:
# def compute_NSE(Q_sim, Q_obs):
#     return 1 - np.sum((Q_sim - Q_obs)**2) / np.sum((Q_obs - np.mean(Q_obs))**2)

In [ ]:
df_attributes = pd.read_csv(Path('data/attributes/polders_data_aangevuld.csv'))
df_attributes.index = df_attributes['SHAPE_ID']

def to_m3(series, key):
    oppervlakte = df_attributes['oppervlak'].loc[key]
    series = series * oppervlakte
    series[series < 0] = 0
    return series

def get_NSE(results_xr_obj, NSE_key='NSE_1D'):
    # NSE_key = 'NSE_1D'
    if NSE_key in results_xr_obj.keys():
        return round(results_xr_obj[NSE_key],2)
    else:
        return np.nan
    
# def get_NSE2(results_xr_obj, key):

#     if 'afvoer_obs' in results_xr_obj['xr'].keys() and 'afvoer_sim' in results_xr_obj['xr'].keys():
#         Q_obs = results_xr_obj['xr']['afvoer_obs'].values
#         Q_sim = results_xr_obj['xr']['afvoer_sim'].values

#         # to m3/s
#         Q_obs = to_m3(Q_obs, key)
#         Q_sim = to_m3(Q_sim, key)

#         # compute NSE
#         NSE = compute_NSE(Q_sim, Q_obs)
#         return round(NSE,2)
    
#     else:
#         return np.nan


In [ ]:
series_keys = list(results_list[0][0].keys())
init_key = series_keys[0]
fig = go.Figure()

# Observed traces (both validation and testing) in blue
fig.add_trace(go.Scatter(
    x=(results_list[0][0][init_key]['1D']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][0][init_key]['1D']['xr']['afvoer_obs'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Geobserveerde validatie data (dagelijks)',
    line=dict(color='blue', dash='dash')
))

fig.add_trace(go.Scatter(
    x=(results_list[0][1][init_key]['1D']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][1][init_key]['1D']['xr']['afvoer_obs'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Geobserveerde testing data (dagelijks)',
    line=dict(color='blue', dash='dash')
))

# Model 1 Gesimuleerde data: validation (light red) and testing (dark red)
fig.add_trace(go.Scatter(
    x=(results_list[0][0][init_key]['1D']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][0][init_key]['1D']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde validatie data (dagelijks), model 1, NSE: {}'.format(get_NSE(results_list[0][0][init_key]['1D'], NSE_key='NSE_1D')),
    line=dict(color='lightsalmon', dash='dash')
))

fig.add_trace(go.Scatter(
    x=(results_list[0][1][init_key]['1D']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][1][init_key]['1D']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde testing data (dagelijks), model 1, NSE: {}'.format(get_NSE(results_list[0][1][init_key]['1D'], NSE_key='NSE_1D')),
    line=dict(color='darkred', dash='dash')
))

# Model 2 Gesimuleerde data: validation (light green) and testing (dark green)
fig.add_trace(go.Scatter(
    x=(results_list[1][0][init_key]['1D']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[1][0][init_key]['1D']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde validatie data (dagelijks), model 2, NSE: {}'.format(get_NSE(results_list[1][0][init_key]['1D'], NSE_key='NSE_1D')),
    line=dict(color='lightgreen', dash='dash')
))

fig.add_trace(go.Scatter(
    x=(results_list[1][1][init_key]['1D']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[1][1][init_key]['1D']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde testing data (dagelijks), model 2, NSE {}'.format(get_NSE(results_list[1][1][init_key]['1D'], NSE_key='NSE_1D')),
    line=dict(color='darkgreen', dash='dash')
))

# Observed traces (both validation and testing) in blue
fig.add_trace(go.Scatter(
    x=(results_list[0][0][init_key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][0][init_key]['1H']['xr']['afvoer_obs'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Geobserveerde validatie data (uurlijks)',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=(results_list[0][1][init_key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][1][init_key]['1H']['xr']['afvoer_obs'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Geobserveerde testing data (uurlijks)',
    line=dict(color='blue')
))

# Model 1 Gesimuleerde data: validation (light red) and testing (dark red)
fig.add_trace(go.Scatter(
    x=(results_list[0][0][init_key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][0][init_key]['1H']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde validatie data (uurlijks), model 1, NSE: {}'.format(get_NSE(results_list[0][0][init_key]['1H'], NSE_key='NSE_1H')),
    line=dict(color='lightsalmon')
))

fig.add_trace(go.Scatter(
    x=(results_list[0][1][init_key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[0][1][init_key]['1H']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde testing data (uurlijks), model 1, NSE: {}'.format(get_NSE(results_list[0][1][init_key]['1H'], NSE_key='NSE_1H')),
    line=dict(color='darkred')
))

# Model 2 Gesimuleerde data: validation (light green) and testing (dark green)
fig.add_trace(go.Scatter(
    x=(results_list[1][0][init_key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[1][0][init_key]['1H']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde validatie data (uurlijks), model 2, NSE: {}'.format(get_NSE(results_list[1][0][init_key]['1H'], NSE_key='NSE_1H')),
    line=dict(color='lightgreen')
))
fig.add_trace(go.Scatter(
    x=(results_list[1][1][init_key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(), 
    y=to_m3(results_list[1][1][init_key]['1H']['xr']['afvoer_sim'].values.reshape(-1), init_key), 
    mode='lines', 
    name='Gesimuleerde testing data (uurlijks), model 2, NSE {}'.format(get_NSE(results_list[1][1][init_key]['1H'], NSE_key='NSE_1H')),
    line=dict(color='darkgreen')
))

dropdown_buttons = []
for key in series_keys:
    # Compute NSE for each model and dataset using the new key:
    nse_model1_val = get_NSE(results_list[0][0][key]['1D'])
    nse_model1_test = get_NSE(results_list[0][1][key]['1D'])
    nse_model2_val = get_NSE(results_list[1][0][key]['1D'])
    nse_model2_test = get_NSE(results_list[1][1][key]['1D'])
    nse_model1_val_u = get_NSE(results_list[0][0][key]['1H'], NSE_key='NSE_1H')
    nse_model1_test_u = get_NSE(results_list[0][1][key]['1H'], NSE_key='NSE_1H')
    nse_model2_val_u = get_NSE(results_list[1][0][key]['1H'], NSE_key='NSE_1H')
    nse_model2_test_u = get_NSE(results_list[1][1][key]['1H'], NSE_key='NSE_1H')
    
    # Create a new names list with the updated NSE values:
    new_names = [
        'Observed validatie data (dagelijks)',
        'Observed testing data',
        f'Gesimuleerde validatie data (dagelijks), model 1, NSE: {nse_model1_val}',
        f'Gesimuleerde testing data (dagelijks), model 1, NSE: {nse_model1_test}',
        f'Gesimuleerde validatie data (dagelijks), model 2, NSE: {nse_model2_val}',
        f'Gesimuleerde testing data (dagelijks), model 2, NSE: {nse_model2_test}',
        'Geobserveerde validatie data (uurlijks)',
        'Geobserveerde testing data (uurlijks)',
        f'Gesimuleerde validatie data (uurlijks), model 1, NSE: {nse_model1_val_u}',
        f'Gesimuleerde testing data (uurlijks), model 1, NSE: {nse_model1_test_u}',
        f'Gesimuleerde validatie data (uurlijks), model 2, NSE: {nse_model2_val_u}',
        f'Gesimuleerde testing data (uurlijks), model 2, NSE: {nse_model2_test_u}'
    ]
    
    dropdown_buttons.append({
        'label': key,
        'method': 'update',
        'args': [
            {
                'x': [
                    results_list[0][0][key]['1D']['xr']['date'].values.reshape(-1),
                    results_list[0][1][key]['1D']['xr']['date'].values.reshape(-1),
                    results_list[0][0][key]['1D']['xr']['date'].values.reshape(-1),
                    results_list[0][1][key]['1D']['xr']['date'].values.reshape(-1),
                    results_list[1][0][key]['1D']['xr']['date'].values.reshape(-1),
                    results_list[1][1][key]['1D']['xr']['date'].values.reshape(-1),

                    (results_list[0][0][key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(),
                    (results_list[0][1][key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(),
                    (results_list[0][0][key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(),
                    (results_list[0][1][key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(),
                    (results_list[1][0][key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten(),
                    (results_list[1][1][key]['1H']['xr']['date'].values.reshape(-1,1) + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten()
                ],
                'y': [
                    to_m3(results_list[0][0][key]['1D']['xr']['afvoer_obs'].values.reshape(-1), key),
                    to_m3(results_list[0][1][key]['1D']['xr']['afvoer_obs'].values.reshape(-1), key),
                    to_m3(results_list[0][0][key]['1D']['xr']['afvoer_sim'].values.reshape(-1), key),
                    to_m3(results_list[0][1][key]['1D']['xr']['afvoer_sim'].values.reshape(-1), key),
                    to_m3(results_list[1][0][key]['1D']['xr']['afvoer_sim'].values.reshape(-1), key),
                    to_m3(results_list[1][1][key]['1D']['xr']['afvoer_sim'].values.reshape(-1), key),

                    to_m3(results_list[0][0][key]['1H']['xr']['afvoer_obs'].values.reshape(-1), key),
                    to_m3(results_list[0][1][key]['1H']['xr']['afvoer_obs'].values.reshape(-1), key),
                    to_m3(results_list[0][0][key]['1H']['xr']['afvoer_sim'].values.reshape(-1), key),
                    to_m3(results_list[0][1][key]['1H']['xr']['afvoer_sim'].values.reshape(-1), key),
                    to_m3(results_list[1][0][key]['1H']['xr']['afvoer_sim'].values.reshape(-1), key),
                    to_m3(results_list[1][1][key]['1H']['xr']['afvoer_sim'].values.reshape(-1), key)
                ],
                # Update the trace names to include the new NSE values:
                'name': new_names
            },
            {'title': f"Time Series for {key}"}
        ]
    })


# Add the dropdown menu to the layout
fig.update_layout(
    title=f"Time Series for {init_key}",
    xaxis_title="Date",
    yaxis_title="Flow",
    updatemenus=[{
        'active': 0,
        'buttons': dropdown_buttons,
        'direction': 'down',
        'showactive': True,
        'x': 1.2,  #
        'y': 1.5,
        'pad': {'r': 20, 't': 20}  # Add padding around the buttons
    }]
)

fig.show()


Hierboven kan je op de legenda klikken om reeksen aan/uit te zetten. Zie het dropdown menu voor welk gebied en zie de knopjes rechtsbovenin om in te zoomen etc.

Let op: de NSE waarden zijn nog berekend van de reeksen die genormaliseerd zijn naar mm/s en niet in m3/s. 

In [ ]:

# NSE scores model 1, dagelijks
NSE_1D_validation_scores_model_1_dagelijks = [get_NSE(results_list[0][0][key]['1D'], NSE_key='NSE_1D') for key in series_keys]
NSE_1D_testing_scores_model_1_dagelijks = [get_NSE(results_list[0][1][key]['1D'], NSE_key='NSE_1D') for key in series_keys]

# NSE scores model 1, uurlijks
NSE_1D_validation_scores_model_1_uurlijks = [get_NSE(results_list[0][0][key]['1H'], NSE_key='NSE_1H') for key in series_keys]
NSE_1D_testing_scores_model_1_uurlijks = [get_NSE(results_list[0][1][key]['1H'], NSE_key='NSE_1H') for key in series_keys]

# NSE scores model 2, dagelijks
NSE_1D_validation_scores_model_2_dagelijks = [get_NSE(results_list[1][0][key]['1D'], NSE_key='NSE_1D') for key in series_keys]
NSE_1D_testing_scores_model_2_dagelijks = [get_NSE(results_list[1][1][key]['1D'], NSE_key='NSE_1D') for key in series_keys]

# NSE scores model 2, uurlijks
NSE_1D_validation_scores_model_2_uurlijks = [get_NSE(results_list[1][0][key]['1H'], NSE_key='NSE_1H') for key in series_keys]
NSE_1D_testing_scores_model_2_uurlijks = [get_NSE(results_list[1][1][key]['1H'], NSE_key='NSE_1H') for key in series_keys]

df_NSE_scores_test_set = pd.DataFrame({
                                # 'NSE_1D_validation_scores_model_1_dagelijks': NSE_1D_validation_scores_model_1_dagelijks,
                                'NSE_1D_testing_scores_model_1_dagelijks': NSE_1D_testing_scores_model_1_dagelijks,
                                # 'NSE_1D_validation_scores_model_2_dagelijks': NSE_1D_validation_scores_model_2_dagelijks,
                                'NSE_1D_testing_scores_model_2_dagelijks': NSE_1D_testing_scores_model_2_dagelijks,
                                # 'NSE_1D_validation_scores_model_1_uurlijks': NSE_1D_validation_scores_model_1_uurlijks,
                                'NSE_1D_testing_scores_model_1_uurlijks': NSE_1D_testing_scores_model_1_uurlijks,
                                # 'NSE_1D_validation_scores_model_2_uurlijks': NSE_1D_validation_scores_model_2_uurlijks,
                                'NSE_1D_testing_scores_model_2_uurlijks': NSE_1D_testing_scores_model_2_uurlijks},
                                 index=series_keys)

df_NSE_scores_validation_set = pd.DataFrame({
                                'NSE_1D_validation_scores_model_1_dagelijks': NSE_1D_validation_scores_model_1_dagelijks,
                                # 'NSE_1D_testing_scores_model_1_dagelijks': NSE_1D_testing_scores_model_1_dagelijks,
                                'NSE_1D_validation_scores_model_2_dagelijks': NSE_1D_validation_scores_model_2_dagelijks,
                                # 'NSE_1D_testing_scores_model_2_dagelijks': NSE_1D_testing_scores_model_2_dagelijks,
                                'NSE_1D_validation_scores_model_1_uurlijks': NSE_1D_validation_scores_model_1_uurlijks,
                                # 'NSE_1D_testing_scores_model_1_uurlijks': NSE_1D_testing_scores_model_1_uurlijks,
                                'NSE_1D_validation_scores_model_2_uurlijks': NSE_1D_validation_scores_model_2_uurlijks},
                                # 'NSE_1D_testing_scores_model_2_uurlijks': NSE_1D_testing_scores_model_2_uurlijks},
                                 index=series_keys)

In [ ]:
df_NSE_scores_validation_set

In [ ]:
df_NSE_scores_test_set

In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import numpy as np

# Assume these variables and functions are defined in your environment:
# series_keys: list of available series (areas)
# results_list: your nested data structure
# to_m3: conversion function

app = dash.Dash(__name__)

app.layout = html.Div([
    html.Div([
        html.Label("Select Area"),
        dcc.Dropdown(
            id="series-dropdown",
            options=[{"label": key, "value": key} for key in series_keys],
            value=series_keys[0]
        )
    ], style={"width": "30%", "display": "inline-block", "verticalAlign": "top"}),
    
    html.Div([
        html.Label("Select Dataset"),
        dcc.Dropdown(
            id="dataset-dropdown",
            options=[
                {"label": "Validation", "value": 0},
                {"label": "Test", "value": 1}
            ],
            value=0
        )
    ], style={"width": "30%", "display": "inline-block", "verticalAlign": "top"}),

    html.Div([
        html.Label("Select Forecast Day"),
        dcc.Dropdown(
            id="forecast-dropdown"
            # Options and value will be set automatically via callback.
        )
    ], style={"width": "30%", "display": "inline-block", "verticalAlign": "top"}),

    dcc.Graph(id="graph")
])

# Callback to update the forecast-dropdown options and its value based on series and dataset.
@app.callback(
    [Output("forecast-dropdown", "options"),
     Output("forecast-dropdown", "value")],
    [Input("series-dropdown", "value"),
     Input("dataset-dropdown", "value")]
)
def update_forecast_dropdown(series_key, dataset_idx):
    # Extract the forecast dates array.
    # Here we assume the dates are stored in results_list[0][dataset_idx][series_key]['1H']['xr']['date']
    # and that it is an array of numpy.datetime64 objects.
    forecast_dates = results_list[0][dataset_idx][series_key]['1H']['xr']['date']
    # Create dropdown options: label is the day as a string (e.g., '2025-03-20') and value is the forecast index.
    new_options = [{"label": str(np.datetime_as_string(date, unit='D')), "value": i} 
                   for i, date in enumerate(forecast_dates)]
    # Set the default forecast value to the first available forecast.
    new_value = new_options[0]["value"] if new_options else None
    return new_options, new_value

# Callback to update the graph based on the selected series, dataset, and forecast.
@app.callback(
    Output("graph", "figure"),
    [Input("series-dropdown", "value"),
     Input("dataset-dropdown", "value"),
     Input("forecast-dropdown", "value")]
)
def update_graph(series_key, dataset_idx, forecast_idx):
    fig = go.Figure()
    
    # Compute x values.
    # We use the selected forecast index to retrieve the base date and then add hourly increments.
    base_date = results_list[0][dataset_idx][series_key]['1H']['xr']['date'][forecast_idx].values
    x = base_date + np.arange(0, 24, 1) * np.timedelta64(1, 'h')
    
    # Get y values for each trace.
    y_obs  = to_m3(results_list[0][dataset_idx][series_key]['1H']['xr']['afvoer_obs'][forecast_idx, :].values, series_key)
    y_sim1 = to_m3(results_list[0][dataset_idx][series_key]['1H']['xr']['afvoer_sim'][forecast_idx, :].values, series_key)
    y_sim2 = to_m3(results_list[1][dataset_idx][series_key]['1H']['xr']['afvoer_sim'][forecast_idx, :].values, series_key)
    
    # Add traces.
    fig.add_trace(go.Scatter(
        x=x, y=y_obs, mode='lines',
        name='Observed data', line=dict(color='blue')
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y_sim1, mode='lines',
        name='Gesimuleerde data, model 1', line=dict(color='darkred')
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y_sim2, mode='lines',
        name='Gesimuleerde data, model 2', line=dict(color='darkgreen')
    ))
    
    # Update layout with a dynamic title.
    # The title now shows the selected forecast day.
    selected_day = np.datetime_as_string(results_list[0][dataset_idx][series_key]['1H']['xr']['date'][forecast_idx].values, unit='D')
    fig.update_layout(
        title=f"Time Series for {series_key} (Dataset: {'Validation' if dataset_idx == 0 else 'Test'}, Day: {selected_day})",
        xaxis_title="Date",
        yaxis_title="Flow"
    )
    
    return fig

if __name__ == '__main__':
    app.run_server(debug=True, port=8051)
